In [11]:
"""
Tracking visualization + segmentation check in napari
─────────────────────────────────────────────────────
  1. TRACKING  — DIC + masks recoloured by TRACK_ID + cell-ID numbers.
  2. SEG CHECK — endpoint fluorescence + last-frame mask outline, which
                 you can shift together over the DIC to align them.

    python view_tracking.py

Alignment keys (window focused):
    arrow keys        nudge fluo + seg together by STEP px
    Shift + arrows    nudge by 1 px (fine)
    r                 reset shift
    p                 print current shift
"""

import os
import glob
import numpy as np
import pandas as pd
import tifffile as tiff
import napari
from napari.utils.colormaps import DirectLabelColormap


# ═══════════════════════════════════════════════════════════════════
# SETTINGS — edit these, then run
# ═══════════════════════════════════════════════════════════════════

BASE = r"batch_output\A07.3_pos35_sc_gaussian8_eq(A07.3_avg41pos_x81t_sc_gaussian8)_med2407"
CSV  = os.path.join(BASE, "results", os.path.basename(BASE) + "_tracked_Overlap.csv")

FLUO_ND2      = r"\\Hive2004\ag_idip\ZeljkaBaca_Data\ACID\250729\A07.3\H7_DENV2_MOI1_40h_fixed_stained.nd2"
#FLUO_ND2      = r"\\Hive2004\ag_idip\ZeljkaBaca_Data\ACID\250618\A04.2\H7_DENV2_MOI1_40h_fixed_stained.nd2"
FLUO_POSITION = 34
CHANNEL_NAMES  = ["DAPI", "membrane", "actin", "viral"]
CHANNEL_COLORS = ["purple", "green", "yellow", "magenta"]

MINFRAMES  = 1
LABEL_SIZE = 5
SHOW_IDS   = True
STEP       = 15        # px per arrow press when aligning

# ═══════════════════════════════════════════════════════════════════
# (DO NOT EDIT BELOW)
# ═══════════════════════════════════════════════════════════════════


def load_stack(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.tif")))
    if not files:
        return None
    return np.stack([tiff.imread(f) for f in files])


def load_fluorescence(path, position):
    import nd2
    with nd2.ND2File(path) as f:
        print(f"  fluo sizes: {f.sizes}")
        img = f.asarray()
    img = img[position]
    if img.ndim == 2:
        return [img]
    return [img[c] for c in range(img.shape[0])]


def main():
    print("loading frames and masks...")
    frames = load_stack(os.path.join(BASE, "frames"))
    masks  = load_stack(os.path.join(BASE, "masks"))
    print(f"  frames: {None if frames is None else frames.shape}")
    print(f"  masks:  {None if masks is None else masks.shape}")

    print("loading tracking table...")
    df = pd.read_csv(CSV)
    if df["POSITION_X"].dtype == object:
        df = pd.read_csv(CSV, skiprows=[1, 2, 3])
    df = df[pd.to_numeric(df["TRACK_ID"], errors="coerce") >= 0].copy()
    for c in ["TRACK_ID", "FRAME", "POSITION_X", "POSITION_Y"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["TRACK_ID", "FRAME", "POSITION_X", "POSITION_Y"])
    if MINFRAMES > 1:
        counts = df.groupby("TRACK_ID")["FRAME"].transform("count")
        df = df[counts >= MINFRAMES]
    df["TRACK_ID"] = df["TRACK_ID"].astype(int)
    df["FRAME"]    = df["FRAME"].astype(int)

    print("recolouring masks by track id...")
    tracked = np.zeros_like(masks)
    T = masks.shape[0]
    for frame_id, g in df.groupby("FRAME"):
        idx = frame_id - 1
        if idx < 0 or idx >= T:
            continue
        m = masks[idx]
        out = np.zeros_like(m)
        for _, row in g.iterrows():
            y, x = int(round(row["POSITION_Y"])), int(round(row["POSITION_X"]))
            if 0 <= y < m.shape[0] and 0 <= x < m.shape[1]:
                cp = m[y, x]
                if cp > 0:
                    out[m == cp] = row["TRACK_ID"] + 1
        tracked[idx] = out

    points = np.column_stack([
        df["FRAME"].to_numpy() - 1,
        df["POSITION_Y"].to_numpy(),
        df["POSITION_X"].to_numpy(),
    ])
    labels = df["TRACK_ID"].astype(str).tolist()

    print("loading fluorescence...")
    try:
        channels = load_fluorescence(FLUO_ND2, FLUO_POSITION)
        print(f"  {len(channels)} channel(s), shape {channels[0].shape}")
    except Exception as e:
        channels = None
        print(f"  ! could not load fluorescence: {e}")

    # ══ viewer ══
    viewer = napari.Viewer(title="Tracking + segmentation check")

    if frames is not None:
        viewer.add_image(frames, name="DIC", colormap="gray")

    # filled tracked cells, coloured by TRACK_ID
    viewer.add_labels(tracked, name="tracked cells", opacity=0.5)

    # separate outline layer of the same tracked cells (toggle independently)
    n_labels = int(tracked.max())
    yellow_cmap = DirectLabelColormap(color_dict={
        **{i: [1.0, 1.0, 0.0, 1.0] for i in range(1, n_labels + 1)},
        None: [0.0, 0.0, 0.0, 0.0],   # pozadina transparentna
    })

    tracked_outline = viewer.add_labels(
        tracked, name="tracked outline", opacity=1.0,
        colormap=yellow_cmap,
    )
    tracked_outline.contour = 2

    if SHOW_IDS:
        viewer.add_points(points, name="cell IDs", size=0,
                          text={"string": labels, "size": LABEL_SIZE,
                                "color": "yellow", "anchor": "center"})

    # fluorescence layers move together during alignment
    aligned = []
    if channels is not None:
        for i, ch in enumerate(channels):
            nm = CHANNEL_NAMES[i] if i < len(CHANNEL_NAMES) else f"ch{i}"
            cl = CHANNEL_COLORS[i] if i < len(CHANNEL_COLORS) else "gray"
            lyr = viewer.add_image(ch, name=f"fluo: {nm}", colormap=cl,
                                   blending="additive", visible=(i == 0))
            aligned.append(lyr)

    # ── alignment: shift fluo + seg together over the DIC ──
    offset = [0, 0]      # (y, x)

    def apply():
        for lyr in aligned:
            lyr.translate = tuple(offset)
        viewer.status = f"fluo/seg shift (y, x) = ({offset[0]}, {offset[1]})"

    def shift(dy, dx):
        offset[0] += dy
        offset[1] += dx
        apply()

    viewer.bind_key("Up",          lambda v: shift(-STEP, 0))
    viewer.bind_key("Down",        lambda v: shift(STEP, 0))
    viewer.bind_key("Left",        lambda v: shift(0, -STEP))
    viewer.bind_key("Right",       lambda v: shift(0, STEP))
    viewer.bind_key("Shift-Up",    lambda v: shift(-1, 0))
    viewer.bind_key("Shift-Down",  lambda v: shift(1, 0))
    viewer.bind_key("Shift-Left",  lambda v: shift(0, -1))
    viewer.bind_key("Shift-Right", lambda v: shift(0, 1))
    viewer.bind_key("r", lambda v: (offset.__setitem__(0, 0), offset.__setitem__(1, 0), apply()))
    viewer.bind_key("p", lambda v: print(f"shift (y, x) = ({offset[0]}, {offset[1]})"))

    # set which layers are visible — THIS is where you control it
    viewer.layers["DIC"].visible = True
    viewer.layers["tracked cells"].visible = True
    viewer.layers["tracked cells"].opacity = 0.05
    viewer.layers["tracked outline"].visible = True
    viewer.layers["tracked outline"].opacity = 0.5
    viewer.layers["cell IDs"].visible = True
    viewer.layers["fluo: ch4"].visible = False
    viewer.layers["fluo: DAPI"].visible = False
    viewer.layers["fluo: viral"].visible = True
    viewer.layers["fluo: actin"].visible = False
    viewer.layers["fluo: membrane"].visible = False

    from napari_animation import Animation

    animation = Animation(viewer)

    # step through every timepoint
    #n_frames = viewer.dims.range[0].stop   # number of frames on the time axis
    #for t in range(int(n_frames)):
    #    viewer.dims.set_point(0, t)         # move time slider to frame t
    #    animation.capture_keyframe()

    #animation.animate(
    #    "tracking_video.avi",
    #    fps=20,                              # 8 frames per second
    #    canvas_only=True,                   # just the image canvas, no napari UI
    #)
    #print("saved tracking_video.avi")

    print("\nAlign: arrows nudge fluo + seg together over the DIC;")
    print("       Shift+arrows = fine (1 px), r = reset, p = print shift.")
    napari.run()


if __name__ == "__main__":
    main()

loading frames and masks...
  frames: (81, 1024, 1024)
  masks:  (81, 1024, 1024)
loading tracking table...
recolouring masks by track id...
loading fluorescence...
  fluo sizes: {'P': 41, 'C': 5, 'Y': 1024, 'X': 1024}
  5 channel(s), shape (1024, 1024)

Align: arrows nudge fluo + seg together over the DIC;
       Shift+arrows = fine (1 px), r = reset, p = print shift.
